**Required imports**

In [1]:
import json
import jmespath
from dotenv import load_dotenv
from groq import Groq

**Load the JSON File**  
Lod the JSON file which has survey data arranged in multi level nesting  

In [2]:
# Load JSON data from file
with open("survey_data.json", "r") as f:
    data = json.load(f)

>Key nesting is used as part of query.  
>Filter done at innesr most level (records)  


In [3]:
# Search for volunteers in Jordan with education "High School"
query = "Jordan.volunteers[?education=='High School']"
results = jmespath.search(query, data)

# Print the filtered records
for person in results:
    print(person)

{'name': 'John Daniel', 'age': 48, 'sex': 'Male', 'education': 'High School', 'annual_earning_usd': 82978.57, 'city': 'Smithshire', 'avg_weekly_work_hours': 49.5, 'location_type': 'Urban'}
{'name': 'Jennifer Thompson', 'age': 37, 'sex': 'Male', 'education': 'High School', 'annual_earning_usd': 95683.59, 'city': 'West Douglasville', 'avg_weekly_work_hours': 39.4, 'location_type': 'Urban'}
{'name': 'Christina Snow', 'age': 24, 'sex': 'Other', 'education': 'High School', 'annual_earning_usd': 38819.56, 'city': 'Andersonbury', 'avg_weekly_work_hours': 31.6, 'location_type': 'Rural'}
{'name': 'Jeremy Turner', 'age': 64, 'sex': 'Male', 'education': 'High School', 'annual_earning_usd': 30972.85, 'city': 'North Maria', 'avg_weekly_work_hours': 22.0, 'location_type': 'Urban'}
{'name': 'Eric Martinez', 'age': 29, 'sex': 'Male', 'education': 'High School', 'annual_earning_usd': 43343.48, 'city': 'Fisherfort', 'avg_weekly_work_hours': 58.4, 'location_type': 'Rural'}
{'name': 'Stephanie Flynn', 'ag

>Loop through at a level, to retain value of that field

In [4]:
# Extract names of urban residents per country
urban_names_per_country = {}

# Loop through the outer dimension
for country, info in data.items():
    #query to get names of volunteers living in urban areas
    query = "volunteers[?location_type=='Urban'].name"
    names = jmespath.search(query, info)
    urban_names_per_country[country] = names

urban_names_per_country


{'Jordan': ['Samuel Larson',
  'John Daniel',
  'Jennifer Thompson',
  'Kathleen Patton',
  'Michelle Rivers',
  'Sean Lane',
  'Jeffrey Heath',
  'Jeremy Turner',
  'Deborah Martinez',
  'Jared Martinez',
  'Brandon Gonzalez',
  'Susan Anderson',
  'Ashley Morgan',
  'Joshua Butler',
  'Stacy Hunter',
  'Brendan Miller',
  'Ms. Sherri Reilly',
  'Tracy Casey',
  'Ashley Spencer',
  'Jake Sanders',
  'Brittany Chen',
  'Christian Sampson',
  'David Park',
  'Daniel Johnson',
  'Stephanie Flynn',
  'Madison Moore',
  'Jonathan Alvarez',
  'Angela Campos',
  'Susan Ritter',
  'Kimberly Weber',
  'Michelle Ball',
  'Joshua Blanchard',
  'Kim White',
  'Kelsey Perkins',
  'Nathaniel Leonard',
  'Beth Carroll',
  'James Alexander',
  'Robin Stevens',
  'Austin Ellis',
  'Catherine Werner',
  'Laura Jones',
  'Kevin Arroyo',
  'Holly Burns',
  'Jeanne Mendoza',
  'Patrick Palmer',
  'Elizabeth Moore',
  'Erin Mcknight',
  'Raven Wheeler',
  'Allen Hooper',
  'Brian Holt',
  'Christian Long',

In [5]:
# Dictionary to hold results
phd_demographics_per_country = {}

# Apply JMESPath filter for each country
for country, info in data.items():
    result = jmespath.search("volunteers[?education=='PhD'][].{age: age, sex: sex}", info)
    if result:  # Only store countries with PhD individuals
        phd_demographics_per_country[country+' - PhD'] = result

phd_demographics_per_country

{'Jordan - PhD': [{'age': 28, 'sex': 'Female'},
  {'age': 42, 'sex': 'Female'},
  {'age': 45, 'sex': 'Male'},
  {'age': 24, 'sex': 'Male'},
  {'age': 27, 'sex': 'Other'},
  {'age': 27, 'sex': 'Other'},
  {'age': 51, 'sex': 'Female'},
  {'age': 48, 'sex': 'Other'},
  {'age': 61, 'sex': 'Female'},
  {'age': 29, 'sex': 'Female'},
  {'age': 58, 'sex': 'Male'},
  {'age': 63, 'sex': 'Male'},
  {'age': 45, 'sex': 'Other'},
  {'age': 59, 'sex': 'Male'},
  {'age': 37, 'sex': 'Female'},
  {'age': 63, 'sex': 'Other'}],
 'Moldova - PhD': [{'age': 25, 'sex': 'Other'},
  {'age': 31, 'sex': 'Male'},
  {'age': 42, 'sex': 'Other'},
  {'age': 22, 'sex': 'Other'},
  {'age': 54, 'sex': 'Male'},
  {'age': 19, 'sex': 'Male'},
  {'age': 38, 'sex': 'Male'},
  {'age': 49, 'sex': 'Other'},
  {'age': 67, 'sex': 'Female'},
  {'age': 58, 'sex': 'Female'},
  {'age': 39, 'sex': 'Male'},
  {'age': 69, 'sex': 'Other'},
  {'age': 60, 'sex': 'Male'},
  {'age': 24, 'sex': 'Male'},
  {'age': 23, 'sex': 'Other'},
  {'age':

**Retrived Data to LLM**  
The data retrieved from JSON is provided to LLM as context  
This provides the relevant data to respond to the prompt  


In [6]:
load_dotenv()
client = Groq()
# client = Groq(api_key="Your Key")

>Specific instuction for Response

In [7]:
R_Instr = "Using the context given, provide response to the user question or statement.\
            Context is provided in JSON format.\
            Provide a comprehensive response"

>The data is being retireved from JSON using list comprehension  


In [8]:
# User prompt
Prompt = "Who is more educated, in each country? Ladies or Gents?"

# Filter out only relevant fields
filtered_data = {
    country: [
        {"sex": v["sex"], "education": v["education"]}
        for v in info["volunteers"]
    ]
    for country, info in data.items()
}

Context = json.dumps (filtered_data)

# Invoke LLM with prompt and context
messages=[
    {
        "role": "system",
        "content": R_Instr
    },

    {
        "role": "user",
        "content":"Context : \n"+ Context
    },

    {
        "role": "user",
        "content": "Query : \n" + Prompt
    }
]
completion = client.chat.completions.create(
    messages=messages,
    model="openai/gpt-oss-120b",
)

print (completion.choices[0].message.content)

**Education‑level comparison (average “score” = No school 0 → High School 1 → Associate 2 → Bachelor 3 → Master 4 → PhD 5)**  

| Country | Average score (Women) | Average score (Men) | Who is, on average, more educated? |
|---------|----------------------|---------------------|-----------------------------------|
| **Jordan** | **≈ 2.41** | ≈ 2.38 | **Ladies** (a small edge) |
| **Moldova** | ≈ 2.26 | **≈ 2.66** | **Gents** |
| **Denmark** | **≈ 2.68** | ≈ 2.39 | **Ladies** |
| **Gibraltar** | ≈ 2.67 | **≈ 3.07** | **Gents** |
| **Morocco** | ≈ 2.69 | **≈ 2.77** | **Gents** (narrow margin) |

**Interpretation**

- **Jordan** and **Denmark** have slightly higher‑educated female populations.  
- **Moldova**, **Gibraltar**, and **Morocco** show a modest advantage for males.  

These conclusions are based on the simple numeric weighting of the education categories listed for each individual in the supplied data.
